In [19]:
import pandas as pd

# Load the dataset
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Unwanted Columns List
unwanted_cols = ['Index']
train_cleaned = train.drop(columns=unwanted_cols, errors='ignore') 
test_cleaned = test.drop(columns=unwanted_cols, errors='ignore') 

# Required Feature List
print("\nThis is the NEW, CLEANED dataframe:")
train_cleaned.head()
test_cleaned.head()


This is the NEW, CLEANED dataframe:


,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN
1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy
2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny
3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy
4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy


In [20]:
#Check for Any NULL Values
print(train_cleaned.isnull().sum())
print(test_cleaned.isnull().sum())

geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64
geohash             0
day                 0
timestamp           0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      1349
Weather           431
dtype: int64


In [21]:
# Create hours and minutes
train[['hour','minute']] = train['timestamp'].str.split(':', expand=True).astype(int)
test[['hour','minute']] = test['timestamp'].str.split(':', expand=True).astype(int)
train.drop('timestamp', axis=1, inplace=True)
test.drop('timestamp', axis=1, inplace=True)

In [22]:
# Create cyclical features
import numpy as np

train['hour_sin'] = np.sin(2*np.pi*train['hour']/24)
train['hour_cos'] = np.cos(2*np.pi*train['hour']/24)

test['hour_sin'] = np.sin(2*np.pi*test['hour']/24)
test['hour_cos'] = np.cos(2*np.pi*test['hour']/24)

In [23]:
# Create rush_hour
train['rush_hour'] = train['hour'].isin(
    [7,8,9,17,18,19]
).astype(int)

test['rush_hour'] = test['hour'].isin(
    [7,8,9,17,18,19]
).astype(int)

In [24]:
def get_period(hour):
    if 5 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 17:
        return 'afternoon'
    elif 17 <= hour < 22:
        return 'evening'
    else:
        return 'night'

In [25]:
# Create period
train['period'] = train['hour'].apply(get_period)
test['period'] = test['hour'].apply(get_period)

In [26]:
# Fill missing values
train['RoadType'] = train['RoadType'].fillna('Unknown')
test['RoadType'] = test['RoadType'].fillna('Unknown')

train['Weather'] = train['Weather'].fillna('Unknown')
test['Weather'] = test['Weather'].fillna('Unknown')

train['Temperature'] = train['Temperature'].fillna(train['Temperature'].median())
test['Temperature'] = test['Temperature'].fillna(test['Temperature'].median())

In [27]:
# Frequency encoding for geohash
freq = train['geohash'].value_counts()

train['geohash_freq'] = train['geohash'].map(freq)
test['geohash_freq'] = test['geohash'].map(freq)

In [28]:
# Target encoding for geohash_target
geo_target = train.groupby('geohash')['demand'].mean()

global_mean = train['demand'].mean()

train['geohash_target'] = train['geohash'].map(geo_target)

test['geohash_target'] = (
    test['geohash']
    .map(geo_target)
    .fillna(global_mean)
)

In [29]:
train['geohash_freq'] = train['geohash_freq'].fillna(0)
test['geohash_freq'] = test['geohash_freq'].fillna(0)

In [30]:
# Target encoding for RoadType_target
target_map = train.groupby(
    'RoadType'
)['demand'].mean()

train['RoadType_target'] = train['RoadType'].map(target_map)
test['RoadType_target'] = test['RoadType'].map(target_map)

In [31]:
# Find null values
print("Train null values:")
print(train.isnull().sum()[train.isnull().sum() > 0])

print("\nTest null values:")
print(test.isnull().sum()[test.isnull().sum() > 0])

Train null values:
Series([], dtype: int64)

Test null values:
Series([], dtype: int64)


In [32]:
#Encode Categorical Columns
categorical_cols = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'period'
]
from sklearn.preprocessing import LabelEncoder

for col in categorical_cols:
    le = LabelEncoder()

    combined = pd.concat([train[col], test[col]], axis=0)

    le.fit(combined.astype(str))

    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

In [33]:
X = train.drop(['demand', 'Index'], axis=1)
y = train['demand']

In [34]:
#Model Validation and Evaluation
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Split into 80% Training and 20% Validation data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42)

In [35]:
test_features = test.drop('Index', axis=1)

In [36]:
from sklearn.metrics import r2_score
# Retrain final model on the complete dataset
final_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
final_model.fit(X, y)

# Run final predictions against the hidden Test Set
test_preds = final_model.predict(test_features)

# Format into standard submission format
submission = pd.DataFrame({'Index': test['Index'], 'demand': test_preds})
submission.to_csv('submission_final.csv', index=False)

rf_preds = model.predict(X_val)
rf_r2 = r2_score(y_val, rf_preds)
print("Random Forest :", rf_r2)

Random Forest : 0.9548508175956223


In [37]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_val)

print("XGBoost R2:", r2_score(y_val, xgb_preds))

XGBoost R2: 0.9513478350109694


In [38]:
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

lgbm_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    random_state=42
)

lgbm_model.fit(X_train, y_train)

lgbm_preds = lgbm_model.predict(X_val)

print("LightGBM R2:", r2_score(y_val, lgbm_preds))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005246 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 970
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 17
[LightGBM] [Info] Start training from score 0.093784
LightGBM R2: 0.9489313123950348


In [39]:
print("Random Forest :", rf_r2)
print("XGBoost       :", r2_score(y_val, xgb_preds))
print("LightGBM      :", r2_score(y_val, lgbm_preds))

Random Forest : 0.9548508175956223
XGBoost       : 0.9513478350109694
LightGBM      : 0.9489313123950348


In [40]:
ensemble_preds = (
    rf_preds +
    xgb_preds +
    lgbm_preds
) / 3

print(
    "Ensemble R2:",
    r2_score(y_val, ensemble_preds)
)

Ensemble R2: 0.9551854797287522


In [41]:
ensemble_preds = (
    0.8 * rf_preds +
    0.1 * xgb_preds +
    0.1 * lgbm_preds
)

print(
    "Weighted Ensemble R2:",
    r2_score(y_val, ensemble_preds)
)

Weighted Ensemble R2: 0.9557831383695046


In [42]:
ensemble_preds = (
    0.9 * rf_preds +
    0.05 * xgb_preds +
    0.05 * lgbm_preds
)
print(
    "Weighted Ensemble R2:",
    r2_score(y_val, ensemble_preds)
)

Weighted Ensemble R2: 0.9554061124940405


In [45]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

rf_final = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

xgb_final = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm_final = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    random_state=42
)

In [46]:
rf_final.fit(X, y)
xgb_final.fit(X, y)
lgbm_final.fit(X, y)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003016 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 971
[LightGBM] [Info] Number of data points in the train set: 77299, number of used features: 17
[LightGBM] [Info] Start training from score 0.093942


LGBMRegressor(learning_rate=0.05, max_depth=8, n_estimators=500,
              random_state=42)

In [47]:
rf_test = rf_final.predict(test_features)
xgb_test = xgb_final.predict(test_features)
lgbm_test = lgbm_final.predict(test_features)

In [48]:
final_preds = (
    0.8 * rf_test +
    0.1 * xgb_test +
    0.1 * lgbm_test
)

In [49]:
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_preds
})

submission.to_csv(
    'submission_ensemble.csv',
    index=False
)

print(submission.head())
print(submission.shape)

   Index    demand
0      0  0.053789
1      1  0.022331
2      2  0.036510
3      3  0.035683
4      4  0.053133
(41778, 2)
